# TerraPulse AI · 01: build the fine-tuning data (Colab)
Open this notebook in VS Code with the **Colab extension** and pick a **T4 GPU** runtime. Nothing here runs on your laptop.

1. intent data: generated with exact labels (`training/gen_intent_data.py`)
2. real fact sheets: the live pipeline (SLM off) on ~160 generated questions (`training/build_compose_data.py`)
3. answer drafts: an **open** teacher (Qwen3-4B, fp16, batched) writes answers from each fact sheet; only drafts that pass
   the runtime **grounding verifier** are kept. No hosted/proprietary LLM is used anywhere.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the project into the Colab runtime. Option A: clone your GitHub repo. Option B: upload GEOAI.zip.
REPO_URL = "https://github.com/aaron43210/GeoAI_Hackathon.git"   # set "" to upload GEOAI.zip instead
import os, subprocess, sys
if REPO_URL and os.path.exists("/content/GEOAI/.git"):
    subprocess.run(["git", "-C", "/content/GEOAI", "pull", "--ff-only"], check=True)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/GEOAI"], check=True)
elif not os.path.exists("/content/GEOAI"):
    from google.colab import files
    up = files.upload()                         # choose GEOAI.zip (project folder zipped, without node_modules)
    subprocess.run(["unzip", "-q", next(iter(up)), "-d", "/content/"], check=True)
%cd /content/GEOAI
!pip -q install -r backend/requirements-core.txt
sys.path.insert(0, "/content/GEOAI/backend")

In [ ]:
!python training/gen_intent_data.py --n 3000 --out training/data/intent.jsonl
!head -c 600 training/data/intent.jsonl

### Real fact sheets from live APIs (≈15–25 min, paced to respect free-API limits)

In [ ]:
!python training/build_compose_data.py --n 160 --pause 2 --out training/data/factsheets.jsonl

### Open teacher drafts → keep only answers that pass the grounding verifier
Batched fp16 Qwen3-4B: about 10–15 min for 160 sheets. Safe to interrupt and re-run: the next cell resumes from `training/data/compose.jsonl`. If CUDA runs out of memory, set `B = 4`.

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate
import json, gc, torch
# If an earlier teacher (e.g. the old Qwen3-8B cell) is still in GPU memory, free it first.
for _name in ("teacher", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
from app.slm.prompts import compose_messages
from app.evidence.grounding import verify

# Open teacher in fp16 with batched generation: ~10-15 min for 160 sheets on a T4.
# (Qwen3-8B in 4-bit was ~9 min per sheet on a T4, far too slow.)
TEACHER = "Qwen/Qwen3-4B"          # Apache-2.0 open weights
tok = AutoTokenizer.from_pretrained(TEACHER, padding_side="left")
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=torch.float16, device_map="cuda")

def draft_batch(sheets, temperature):
    texts = [tok.apply_chat_template(compose_messages(s), tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False) for s in sheets]
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = teacher.generate(**enc, max_new_tokens=320, do_sample=True, temperature=temperature,
                               top_p=0.9, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

In [ ]:
import os
rows = [json.loads(l) for l in open("training/data/factsheets.jsonl")]
# Keep answers from an interrupted earlier run that are still in memory but not yet on disk.
_mem = list(globals().get("kept") or [])
_disk = [json.loads(l) for l in open("training/data/compose.jsonl")] if os.path.exists("training/data/compose.jsonl") else []
_seen = {k["messages"][1]["content"] for k in _disk}
_extra = [k for k in _mem if k["messages"][1]["content"] not in _seen]
if _extra:
    with open("training/data/compose.jsonl", "a") as fh:
        for k in _extra: fh.write(json.dumps(k, ensure_ascii=False) + "\n")
    print("saved", len(_extra), "answers from the interrupted run")
kept, stats = [], {"teacher": 0, "template": 0, "dropped": 0}
if os.path.exists("training/data/compose.jsonl"):           # resume after an interruption
    kept = [json.loads(l) for l in open("training/data/compose.jsonl")]
    for k in kept: stats[k["source"]] += 1
done = {k["messages"][1]["content"] for k in kept}
todo = [r for r in rows if r["fact_sheet"] not in done]
print(len(kept), "kept so far,", len(todo), "to go")

def ok(r, a):
    return verify(a, r["fact_sheet"], set(r["evidence_ids"]), r["known_names"], r["must_mention"],
                  r["recommended"], r.get("evidence_values")).status == "passed"

def save(r, src, ans):
    msgs = compose_messages(r["fact_sheet"]) + [{"role": "assistant", "content": ans}]
    item = {"task": "compose", "source": src, "messages": msgs}
    kept.append(item); stats[src] += 1
    with open("training/data/compose.jsonl", "a") as fh: fh.write(json.dumps(item, ensure_ascii=False) + "\n")

B = 8                                   # lower to 4 if CUDA runs out of memory
for i in range(0, len(todo), B):
    batch = todo[i:i + B]
    retry = []
    for r, a in zip(batch, draft_batch([r["fact_sheet"] for r in batch], 0.5)):
        save(r, "teacher", a) if ok(r, a) else retry.append(r)
    if retry:
        for r, a in zip(retry, draft_batch([r["fact_sheet"] for r in retry], 0.8)):
            if ok(r, a): save(r, "teacher", a)
            elif stats["template"] < 0.25 * len(rows): save(r, "template", r["template_answer"])
            else: stats["dropped"] += 1
    print(f"{min(i + B, len(todo))}/{len(todo)}", stats)
print("final", stats, "kept", len(kept))

In [ ]:
# Hugging Face login: uses the Colab secret HF_TOKEN if available, otherwise asks (input is hidden, never saved).
import os
from huggingface_hub import login
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    from getpass import getpass
    token = getpass("Hugging Face write token: ")
login(token)
os.environ["HF_TOKEN"] = token
HF_USER = "aaronr43210"

In [ ]:
# Optional: publish the datasets (open licence) so the model card can link them
from huggingface_hub import HfApi
api = HfApi(); repo = f"{HF_USER}/terrapulse-sft-data"
api.create_repo(repo, repo_type="dataset", exist_ok=True)
for f in ["intent.jsonl", "compose.jsonl", "factsheets.jsonl"]:
    api.upload_file(path_or_fileobj=f"training/data/{f}", path_in_repo=f, repo_id=repo, repo_type="dataset")
print("https://huggingface.co/datasets/" + repo)